# Day05 · 배포한 MCP 서버를 노트북에서 호출 (LLM이 RAG 실행)
`day05/deploy/server.py` 를 **Horizon(FastMCP Cloud)** 에 배포한 뒤, 그 **공개 URL**에 붙어 실습합니다.
- 실습의 Lab 7과 **유일한 차이 = `Client(mcp)` → `Client(MCP_URL)`** (인메모리 → 배포 URL)
- 배포 방법은 `day05/deploy/README.md` 참고
> 노트북은 `asyncio.run()` 없이 **셀에서 그냥 `await`** 를 씁니다.

## 셀 1 · 설치 + 설정
`MCP_URL` 을 배포받은 주소로 바꾸세요. 토큰은 입력창으로 받습니다(하드코딩 금지).

In [ ]:
%pip install -q fastmcp openai

import os, json, getpass
from openai import OpenAI
from fastmcp import Client

MCP_URL = "https://<이름>.fastmcp.app/mcp"          # ← Horizon 배포 URL로 교체
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or getpass.getpass("NVIDIA 토큰(nvapi-...): ")
llm = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)
LLM_MODEL = "qwen/qwen3-next-80b-a3b-instruct"     # non-thinking

## 셀 2 · 배포 서버에 붙어 도구 '직접' 호출 (LLM 없이 확인)
먼저 URL 연결·도구 목록·검색이 되는지 눈으로 확인.

In [ ]:
async with Client(MCP_URL) as c:                    # ← 배포된 원격 URL에 연결
    print("도구:", [t.name for t in await c.list_tools()])
    hits = (await c.call_tool("search_docs", {"query": "연차 신청 며칠 전", "k": 2})).data
    print("검색 결과:", [h["doc_id"] for h in hits])

## 셀 3 · LLM(Qwen)이 원격 도구를 '스스로' 호출 = RAG 실행
Qwen이 배포된 `search_docs` 를 골라 부르고, 그 근거로 답합니다.

In [ ]:
def to_openai_tools(mcp_tools):
    return [{"type": "function", "function": {
        "name": t.name, "description": t.description or "", "parameters": t.inputSchema}} for t in mcp_tools]

async def run_agent(question, max_steps=4):
    async with Client(MCP_URL) as client:           # ← 배포 URL
        oai = to_openai_tools(await client.list_tools())
        messages = [{"role": "system", "content": "사내 문서 도구로 근거를 찾아 한국어로 간결히 답하라."},
                    {"role": "user", "content": question}]
        for _ in range(max_steps):
            m = llm.chat.completions.create(
                model=LLM_MODEL, messages=messages, tools=oai, temperature=0.2, max_tokens=400,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},   # thinking 끄기
            ).choices[0].message
            if not m.tool_calls:
                return m.content
            messages.append({"role": "assistant", "content": m.content or "",
                             "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
            for tc in m.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"  [LLM → 원격 도구] {tc.function.name}({args})")
                res = await client.call_tool(tc.function.name, args)   # 원격 서버가 RAG 실행
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(res.data, ensure_ascii=False, default=str)})
        return "(반복 한도 초과)"

# 노트북은 asyncio.run 없이 그냥 await
print("A:", await run_agent("연차는 며칠 전에 신청해야 해?"))